# Thesis Note 03.1 — Data Pipeline Architecture

## Milestone

Milestone 03 — Data Pipeline Design

Sub-Milestone 03.1 — Data Pipeline Architecture

---

# Objective

The objective of this milestone is to define the overall architecture of the data pipeline before any implementation begins. A well-designed data pipeline serves as the foundation for all subsequent stages of the project, including baseline models, self-supervised learning, multimodal fusion, and missing-modality experiments.

Instead of developing task-specific preprocessing code, the pipeline is designed to remain reusable, reproducible, and independent of any particular learning algorithm.

---

# Motivation

The COde dataset contains multiple levels of hierarchy:

* Patient
* Visit (Checkup)
* Images
* Clinical Text

In addition, previous audit studies revealed several important characteristics of the dataset:

* Multiple visits may belong to the same patient.
* Images can be reused across different visits of the same patient.
* Radiographs are naturally missing for a substantial proportion of visits.
* Clinical text is available for nearly every visit.
* A visit may contain multiple photographs and multiple radiographs.

These observations imply that the pipeline must preserve the hierarchical structure of the dataset while preventing information leakage between training and evaluation sets.

---

# Design Objectives

The proposed data pipeline should satisfy the following objectives:

* Fully reproducible
* Config-driven
* Modular
* Independent from downstream models
* Independent from downstream learning tasks
* Compatible with naturally missing modalities
* Compatible with patient-level data partitioning
* Easily extendable for future experiments

---

# Architecture Decisions

## AD-01 — Patient-Level Split

The patient is selected as the unit of dataset partitioning.

All visits belonging to a patient must remain in the same split (training, validation, or testing).

This decision prevents both patient-level information leakage and cross-visit duplicate image leakage.

Status:

Approved

---

## AD-02 — Visit as the Fundamental Sample

Although the split is performed at the patient level, the basic learning sample is defined at the visit level.

Each visit represents one multimodal observation of a patient at a particular time point.

Status:

Approved

---

## AD-03 — Multimodal Sample Representation

Each visit is represented as a single multimodal sample consisting of:

* Patient identifier
* Visit identifier
* Photograph paths
* Radiograph paths
* Clinical text
* Metadata
* Missing-modality indicators
* Raw labels

The pipeline therefore treats all available information belonging to a visit as one coherent sample.

Status:

Approved

---

## AD-04 — Model-Agnostic Design

The data pipeline must not contain any assumptions regarding the downstream learning model.

It should support conventional CNNs, Vision Transformers, multimodal encoders, self-supervised learning frameworks, and future architectures without modification.

Status:

Approved

---

## AD-05 — Task-Agnostic Design

The pipeline must remain independent of downstream prediction tasks.

Tasks such as diagnosis classification, multi-label prediction, retrieval, self-supervised learning, or missing-modality prediction should not influence the data loading process.

Task-specific processing will be implemented in a dedicated future milestone.

Status:

Approved

---

## AD-06 — Preserve Naturally Missing Modalities

Missing radiographs represent a genuine characteristic of the COde dataset rather than corrupted data.

Therefore:

* Missing samples must not be removed.
* Artificial imputation is not performed during data loading.
* Missing information is explicitly represented using modality flags.

This design enables future research on naturally incomplete multimodal learning.

Status:

Approved

---

## AD-07 — Deferred Label Processing

Diagnostic labels remain in their original form during the data pipeline stage.

Operations such as:

* label normalization
* synonym merging
* class filtering
* long-tail handling
* target definition

will be performed later during the Task Definition & Diagnostic Processing milestone.

Status:

Approved

---

## AD-08 — Single Responsibility Principle

Each component of the pipeline performs one clearly defined responsibility.

Examples include:

* CSV loading
* split assignment
* image loading
* clinical text loading
* metadata extraction
* missing-modality handling
* PyTorch dataset construction

This modular design simplifies maintenance and future extensions.

Status:

Approved

---

# Logical Data Hierarchy

The logical organization of the dataset is defined as follows:

Patient

↓

Visit

↓

Multimodal Sample

↓

Modalities

Each visit corresponds to exactly one multimodal sample.

The modalities associated with a visit may include:

* Clinical text
* Photographs
* Radiographs

depending on their availability.

---

# Data Flow

The proposed processing pipeline is summarized below.

COde Dataset

↓

Load CSV

↓

Load Patient-Level Split

↓

Assign Dataset Split

↓

Visit Records

↓

Build Multimodal Samples

↓

Image Loader

Clinical Text Loader

Metadata Loader

↓

Missing-Modality Handler

↓

PyTorch Dataset

↓

Transforms

↓

DataLoader

---

# Component Responsibilities

The pipeline is divided into independent components.

CSV Loader

Reads the raw dataset.

Split Loader

Loads patient-level split assignments.

Sample Builder

Constructs multimodal samples from visit records.

Image Loader

Loads image files and validates file availability.

Clinical Text Loader

Extracts clinical narratives.

Metadata Loader

Extracts auxiliary structured information.

Missing-Modality Handler

Computes explicit modality availability indicators.

Dataset

Provides samples for PyTorch.

Transforms

Applies image and text preprocessing.

DataLoader

Constructs mini-batches for model training.

---

# Expected Benefits

The proposed architecture provides several advantages.

* Prevents patient-level data leakage.
* Supports naturally missing modalities.
* Separates data processing from model implementation.
* Enables reproducible experimentation.
* Simplifies integration of multiple learning paradigms.
* Facilitates future extension without major refactoring.
* Provides a consistent data interface across all experiments.

---

# Relation to Subsequent Milestones

This architectural design serves as the foundation for the remaining implementation stages.

The following milestones will progressively implement the proposed design:

* Configuration System
* Multimodal Sample Representation
* Image Loading
* Clinical Text Processing
* Missing-Modality Handling
* PyTorch Dataset
* DataLoader Factory
* Pipeline Validation

No implementation is performed during this milestone.

The outcome of this stage is the finalized architectural specification that will guide all future development.


# Thesis Note 03.2 — Configuration System Design

## Milestone

Milestone 03 — Data Pipeline Design

Sub-Milestone 03.2 — Configuration System

---

# Objective

The objective of this milestone is to establish a lightweight and reproducible configuration system for the project pipeline.

The purpose of introducing configuration files is to separate implementation logic from experiment settings and environment-specific parameters.

The configuration system enables:

* reproducible experiments,
* easier hardware adaptation,
* reduced hard-coded parameters,
* simpler future extensions.

---

# Motivation

The project is designed to run initially on a local development environment with limited computational resources:

* NVIDIA RTX 3050 Laptop GPU
* 4GB VRAM
* WSL2 environment

However, future experiments may be executed on a more powerful server GPU.

Therefore, the pipeline requires a mechanism that allows changing execution environments without modifying the source code.

---

# Design Principle

The configuration system follows an incremental architecture strategy.

Instead of defining all possible future configurations at the beginning, only configurations required for the current development stage are introduced.

Future configuration files will be added when corresponding implementation components become necessary.

This prevents unnecessary complexity and avoids premature abstraction.

---

# Configuration Structure

The current configuration structure is:

```
configs/
│
├── audit.yaml
├── base.yaml
├── local_rtx3050.yaml
└── server.yaml
```

---

# Base Configuration

File:

```
configs/base.yaml
```

The base configuration contains shared project-level settings.

It defines:

* random seed,
* reproducibility settings,
* dataset paths,
* output directories,
* dataset hierarchy decisions,
* runtime defaults.

Current design:

```yaml
project:
  seed: 42
  deterministic: true

paths:
  dataset_csv: data/raw/COde-Dataset/complete_dataset.csv
  images_root: data/raw/COde-Dataset/Images
  patient_split: results/patient_level_split/patient_split.csv
  output_dir: results

dataset:
  sample_unit: visit
  split_unit: patient

runtime:
  device: auto
  num_workers: 4
  pin_memory: true
```

---

# Local RTX 3050 Configuration

File:

```
configs/local_rtx3050.yaml
```

The local configuration contains only values that differ from the base configuration.

This configuration targets development on:

* RTX 3050 Laptop GPU
* 4GB VRAM

Current overrides:

```yaml
runtime:
  device: cuda
  num_workers: 2
  pin_memory: true
```

The configuration intentionally avoids adding training-related parameters because model training has not started yet.

---

# Server Configuration

File:

```
configs/server.yaml
```

The server configuration is currently a placeholder.

No assumptions are made regarding:

* GPU model,
* VRAM capacity,
* CPU resources,
* storage system.

Only the expected CUDA execution environment is specified.

Future server-specific parameters will be added after the actual hardware configuration becomes available.

---

# Architecture Decisions

## AD-09 — Incremental Architecture & Configuration Design

Status:

Approved

Decision:

The project architecture and configuration system will evolve incrementally.

Only components required by the current milestone are implemented.

Rationale:

This project is research-oriented, and experimental requirements may change during development.

Prematurely designing all future modules increases unnecessary complexity and creates additional refactoring effort.

---

## AD-10 — Minimal Override Configuration

Status:

Approved

Decision:

Environment-specific configuration files should contain only parameters that differ from the base configuration.

Common values should remain in the base configuration.

Rationale:

This avoids duplicated configuration values and reduces inconsistency between different execution environments.

---

# Current Configuration Scope

At this stage, the configuration system covers:

* dataset paths,
* split paths,
* output paths,
* reproducibility settings,
* runtime environment.

It intentionally does not include:

* model parameters,
* optimizer settings,
* training parameters,
* SSL parameters,
* evaluation settings.

These will be introduced only when their corresponding implementation stages begin.

---

# Relation to Future Milestones

The configuration system will later be extended to support:

* dataset loading,
* image preprocessing,
* multimodal fusion,
* model selection,
* self-supervised learning,
* training experiments,
* evaluation protocols.

The current design provides the minimal foundation required for the next milestone:

Milestone 10.3 — Multimodal Sample Representation.

---

# Conclusion

A lightweight configuration framework was established to support reproducible and scalable development.

The current design separates environment-specific settings from implementation logic while maintaining flexibility for future research experiments.


# Thesis Note 03.3 — Multimodal Sample Representation Design

## Milestone

Milestone 03 — Data Pipeline

Sub-Milestone 03.3 — Multimodal Sample Representation

---

# Objective

The objective of this milestone was to define the fundamental data unit used throughout the multimodal learning pipeline.

Before implementing dataset loading and model training, a clear representation of each data sample was required to ensure:

* reproducible data handling,
* consistent interaction between pipeline components,
* explicit management of missing modalities,
* future compatibility with supervised and self-supervised learning experiments.

---

# Dataset Unit Definition

Based on previous dataset analysis and leakage investigation, the following decisions were adopted:

## Split Unit

Patient-level splitting.

All visits belonging to the same patient must remain inside the same partition.

Partitions:

* Train
* Validation
* Test

---

## Sample Unit

Visit-level representation.

Each sample corresponds to one dental checkup:

```
Patient
 ├── Visit 1  → Sample 1
 ├── Visit 2  → Sample 2
 └── Visit 3  → Sample 3
```

The patient identifier is preserved inside each sample to maintain traceability and enforce correct partitioning.

---

# Multimodal Sample Schema

A unified sample representation was designed:

```
MultimodalSample

├── patient_id
├── visit_id
├── split
│
├── photographs
├── radiographs
├── clinical_text
│
├── metadata
├── labels
│
└── missing_flags
```

---

# Identifier Information

Each sample contains:

* patient identifier
* visit/checkup identifier

Example:

```
patient_id = 0001
visit_id = 003
```

These fields allow:

* patient-level split verification,
* longitudinal analysis,
* debugging and traceability.

---

# Image Modality Representation

Both image modalities are represented as lists.

## Photographs

A visit may contain multiple intraoral photographs.

Representation:

```
photographs = [
    image_1,
    image_2,
    ...
]
```

---

## Radiographs

A visit may contain zero, one, or multiple radiographic images.

Representation:

```
radiographs = [
    image_1,
    image_2,
    ...
]
```

Using lists instead of single paths preserves the original multimodal structure of the COde dataset.

---

# Clinical Text Representation

Clinical information is stored as a dictionary instead of a single string.

Example:

```
clinical_text = {

    "diagnosis": "...",

    "examination": "...",

    "treatment_plan": "..."

}
```

This design preserves individual clinical fields and allows future experiments with different text encoding strategies.

---

# Missing Modality Representation

Because naturally missing radiographs are a central research aspect, missingness is explicitly stored.

Example:

```
missing_flags = {

    "has_photographs": True,

    "has_radiographs": False,

    "has_clinical_text": True

}
```

This prevents ambiguity between:

* missing modality,
* empty data,
* unavailable file.

---

# Implementation

A Python dataclass implementation was created:

```
src/data/sample.py
```

The class:

```
MultimodalSample
```

provides:

* structured storage,
* type consistency,
* default handling for optional fields,
* utility functions for modality checking.

---

# Validation Layer

A separate validation module was implemented:

```
src/data/sample_validation.py
```

The validator checks:

## Identity consistency

* patient_id existence
* visit_id existence

## Modality consistency

* modality availability
* missing flag agreement

## Clinical text consistency

* presence/absence consistency

## Optional file validation

* image path existence

---

# Design Decision

## AD-11 — Visit-Level Multimodal Sample Representation

Status:

Approved

Decision:

Each dataset sample is represented as one patient visit containing all available modalities.

Rationale:

The COde dataset is naturally organized around dental checkups. Maintaining visit-level samples preserves the relationship between:

* clinical information,
* photographs,
* radiographs,
* missing modality patterns.

---

## AD-12 — Explicit Missing Modality Encoding

Status:

Approved

Decision:

Missing modalities are represented explicitly through dedicated flags.

Rationale:

Naturally missing radiographs are a key research component. Missingness must be treated as a data property rather than an implementation detail.

---

# Current Pipeline Flow

After this milestone, the planned pipeline becomes:

```
COde CSV
   |
   v
Patient-Level Split
   |
   v
Sample Builder
   |
   v
MultimodalSample
   |
   v
Sample Validation
   |
   v
Image/Text Loading
   |
   v
PyTorch Dataset
   |
   v
DataLoader
```

---

# Conclusion

The multimodal sample representation layer was successfully designed and implemented.

The project now has a stable data contract between raw dataset processing and future learning components.

The next milestone focuses on implementing image loading while supporting:

* multiple images per visit,
* photographs,
* radiographs,
* missing files,
* future PyTorch integration.


# Thesis Note 04 — Image Loading & Validation Pipeline

## Overview

This stage implements the image handling layer of the multimodal data pipeline for the COde dental dataset.

The objective of this component is to provide a unified and reproducible mechanism for:

- loading dental images from different modalities,
- handling multiple images associated with a single visit,
- managing naturally missing image information,
- validating image availability and integrity.

The implemented pipeline supports both visual modalities available in COde:

- Photographs
- Radiographs

This component connects the Sample Representation layer with future PyTorch Dataset/DataLoader implementations.

---

# Relation to Multimodal Sample Representation

In the previous stage (03.3 — Multimodal Sample Representation), each sample was defined as:
Sample
|
├── patient_id
├── visit_id
├── photographs
├── radiographs
├── clinical_text
├── metadata
├── labels
└── missing_flags


The current stage processes the image components:


Sample
|
├── photographs
|
└── radiographs

and converts image references into machine learning compatible tensor representations.

---

# Pipeline Design

The implemented data flow is:


Dataset CSV
|
v
Image References
|
v
Image Loader
|
v
Tensor Representation
|
v
Image Validation
|
v
Validated Multimodal Sample


The image layer is intentionally separated from:

- preprocessing,
- augmentation,
- normalization,
- feature extraction,
- multimodal fusion.

These operations will be implemented in later pipeline stages.

---

# Image Loader Component

## Implementation

The ImageLoader component is implemented as:


src/data/image_loader.py


The responsibility of this module is limited to:

- reading image files,
- converting images into PyTorch tensors,
- handling invalid or unavailable image paths.

The loader does not perform:

- resizing,
- normalization,
- augmentation,
- feature extraction.

---

# Supported Modalities

## Photographs

Photographs represent the visual appearance of dental structures and oral cavity conditions.

Examples:

- intraoral photographs,
- dental surface images,
- clinical oral images.

Dataset statistics:


Photographs checked:
49938 images


Photographs represent the dominant visual modality in the COde dataset.

---

## Radiographs

Radiographs represent structural dental information.

Examples:

- panoramic radiographs,
- dental X-rays,
- radiographic examinations.

Dataset statistics:


Radiographs checked:
8056 images


Radiographs have naturally incomplete availability across visits.

This missingness is preserved because it represents an important characteristic for future missing-modality experiments.

---

# Multiple Image Handling

A single patient visit may contain multiple images.

Example:


Visit Sample

photographs:

[
image_1,
image_2,
image_3
]

radiographs:

[
image_4,
image_5
]


The ImageLoader accepts a list of image paths and loads every available image independently.

This design supports future aggregation strategies such as:

- image-level pooling,
- attention-based aggregation,
- multiple-instance learning,
- modality-specific encoders.

---

# Missing Image Handling

The COde dataset contains naturally incomplete multimodal samples.

The implemented strategy is:

- existing images are loaded,
- missing files are detected,
- invalid images are skipped,
- pipeline execution continues without interruption.

Missing modality information is not removed.

Instead, it is preserved through the Sample Representation layer:


missing_flags


This allows later experiments on:

- missing radiograph robustness,
- incomplete multimodal learning,
- modality dropout scenarios.

---

# Tensor Representation

Each loaded image is converted into a PyTorch tensor.

The current conversion pipeline:


Image File
|
v
RGB Conversion
|
v
PyTorch Tensor
|
v
[C, H, W]


Example validation:

Input:


data/raw/COde-Dataset/Images/Photographs/1139-001-01.jpg


Output:


torch.Size([3, 448, 298])


The original image resolution is preserved.

Image resizing and normalization are intentionally postponed to the preprocessing stage.

---

# Image Validation Component

## Implementation

The validation module is implemented as:


src/data/image_validation.py


The objective of this module is to verify:

- image file existence,
- image readability,
- modality-specific availability,
- corrupted image detection.

---

# Validation Experiment

Command:


python -m src.data.image_validation --force


Dataset:


COde Dataset

Visits:
8775


Validation results:


Total images checked:
57994

Photographs checked:
49938

Radiographs checked:
8056

Missing files:
0

Unreadable files:
0


Generated outputs:


results/image_validation/

├── image_validation_summary.json
├── photograph_validation.csv
├── radiograph_validation.csv
└── corrupted_images.csv


---

# Validation Conclusion

The validation process confirmed that:

- all referenced image files exist,
- all images are readable,
- no corrupted images were detected,
- both image modalities are correctly accessible.

Therefore, the COde image repository is ready for integration into the next pipeline stages.

---

# Reproducibility

The image handling layer follows a modular architecture:


src/data/

├── sample.py
├── sample_validation.py
├── image_loader.py
└── image_validation.py


This structure provides:

- independent testing,
- reproducible experiments,
- reusable dataset components,
- easier integration with PyTorch Dataset classes.

---

# Design Decisions

## Image preprocessing is postponed

At this stage:

- resizing is not applied,
- normalization is not applied,
- augmentation is not applied.

Reason:

These operations depend on:

- selected backbone architecture,
- GPU limitations,
- experiment configuration.

They will be introduced in later preprocessing stages.

---

## Missing modality information is preserved

Missing radiographs are not considered errors.

They represent real-world incomplete multimodal scenarios.

Therefore, missingness information is preserved for future:

- multimodal robustness experiments,
- missing-modality analysis,
- ablation studies.

---

# Current Status

Completed:

✅ Image Loading  
✅ Multiple image support  
✅ Photographs support  
✅ Radiographs support  
✅ Missing image handling  
✅ Image validation  
✅ Corrupted image detection  

---

# Next Step

The next pipeline component is:

## 03.5 — Clinical Text Processing & Metadata Handling

Objectives:

- loading clinical text information,
- cleaning textual fields,
- preparing text modality representation,
- integrating metadata into multimodal samples.

# Thesis Note 03.5 — Clinical Text Processing Pipeline

## Overview

This stage implements the clinical text processing component of the multimodal data pipeline for the COde dental dataset.

The objective of this component is to extract, organize, and validate clinical textual information associated with each patient visit.

The clinical text modality contains multiple medical fields describing patient history, examination findings, radiographic observations, diagnosis-related information, and treatment information.

The implemented text processing layer prepares these fields for future integration with language models and multimodal representation learning frameworks.

---

# Motivation

The COde dataset provides structured clinical information in addition to image modalities.

Each patient visit may contain multiple textual descriptions including:

* patient history,
* chief complaint,
* present illness,
* medical records,
* clinical examination,
* radiographic examination,
* treatment information.

However, clinical text availability is naturally incomplete.

Therefore, the pipeline must:

* preserve missing text information,
* avoid removing incomplete samples,
* provide consistent text extraction,
* prepare data for future tokenizer-based processing.

Missing clinical information is considered part of the dataset characteristics rather than an error.

---

# Clinical Text Fields

The current text processing pipeline uses the following English clinical fields:


patient_record
chief_complaint
present_illness
past_medical_record
examination
radiographs_examination
treatment_plan
treatment_recommendations
management
medical_instructions
remarks
anomalies_en


These fields represent different aspects of patient visits and will later support multimodal learning experiments.

---

# Design Decision

The architecture separates text loading from later language processing components.

The current design follows:


Dataset Row
|
v
Clinical Text Loader
|
v
Structured Text Dictionary
|
v
Tokenizer / Text Encoder (Future Stage)


The ClinicalTextLoader is responsible for:

* extracting selected clinical fields,
* maintaining field names,
* handling missing values.

The loader does not perform:

* tokenization,
* embedding generation,
* text augmentation,
* language model encoding.

These operations will be implemented in later stages.

---

# Text Representation

Each clinical sample is represented as a dictionary:


{
"patient_record": "...",
"chief_complaint": "...",
"present_illness": "...",
"examination": "...",
"treatment_plan": "..."
}


This representation preserves the original structure of clinical information and allows flexible usage with different text encoders.

Future experiments can replace this representation with:

* transformer-based embeddings,
* clinical language models,
* contrastive text-image representations.

---

# Missing Text Handling

Clinical text missingness is preserved during processing.

The pipeline does not:

* remove visits with missing text,
* replace missing fields with artificial content,
* merge different clinical sections.

Instead:

* missing values are detected,
* missing statistics are recorded,
* original dataset structure is preserved.

This approach is important for studying incomplete multimodal learning scenarios.

---

# Validation Strategy

A validation module was implemented to analyze clinical text availability.

The validation checks:

* number of available values,
* number of missing values,
* missing percentage,
* minimum text length,
* maximum text length,
* average text length,
* median text length.

Validation was performed on all 8775 patient visits.

---

# Validation Results

Dataset:


COde Dataset

Total visits:
8775

Validated clinical fields:
12


Overall clinical text statistics:


Total missing values:
36022

Overall missing rate:
34.21%


The results confirm that clinical text availability is heterogeneous across different fields.

---

# Field-Level Missingness Analysis

Important observations:

| Field | Missing Rate |
|---|---:|
| patient_record | 0.24% |
| examination | 2.36% |
| chief_complaint | 5.69% |
| present_illness | 18.37% |
| past_medical_record | 51.50% |
| radiographs_examination | 76.28% |
| treatment_plan | 58.34% |
| treatment_recommendations | 85.79% |

The high missing rate in radiographic examination text is consistent with the naturally incomplete availability of radiographic images.

Therefore, both visual and textual radiographic information represent incomplete modalities in the dataset.

---

# Implementation

The clinical text processing module is implemented as:


src/data/text_loader.py
src/data/text_validation.py


The validation outputs are stored in:


results/text_validation/


Generated files:


text_validation_summary.json
field_statistics.csv
missing_values.csv


---

# Reproducibility Considerations

The text processing pipeline is implemented as an independent component.

This design enables:

* reproducible preprocessing,
* independent validation,
* future integration with Dataset and DataLoader classes,
* flexible replacement of text encoders.

No experiment-specific assumptions are introduced at this stage.

---

# Limitations

At this stage:

* tokenization is not implemented,
* text embeddings are not generated,
* clinical terminology normalization is not applied,
* diagnosis labels are not processed.

These components are intentionally postponed to later milestones.

---

# Next Step

The next stage will focus on completing the multimodal sample preparation pipeline.

Future steps include:

* integration of image and text representations,
* dataset object implementation,
* modality availability tracking,
* label and diagnosis processing.

The current pipeline provides a validated foundation for multimodal self-supervised learning experiments on the COde dataset.

# Thesis Note 03.6 — Missing-Modality Handling Pipeline

## Overview

This stage implements the missing-modality handling component of the multimodal data pipeline for the COde dental dataset.

The objective of this component is to explicitly represent naturally missing modalities without:

* removing incomplete samples,
* performing data imputation,
* generating artificial modality information.

The implemented approach preserves missingness as an inherent characteristic of the dataset and prepares the pipeline for future missing-modality learning experiments.

---

# Motivation

The COde dataset contains multiple heterogeneous modalities:

* Photographs
* Radiographs
* Clinical text

However, these modalities are not available for every patient visit.

Examples:

* A visit may contain intraoral photographs but no radiographs.
* A visit may contain radiographs but incomplete clinical descriptions.
* Some clinical fields may be unavailable.

Removing these samples would reduce dataset diversity and introduce selection bias.

Therefore, missing modalities are treated as meaningful information rather than invalid samples.

---

# Design Decision

The pipeline follows the principle:


Original Sample
|
v
Modality Availability Check
|
v
Missing Flags
|
v
Multimodal Sample Representation


The missing modality component only identifies availability status.

It does not:

* fill missing values,
* synthesize missing images,
* predict missing modalities,
* discard incomplete samples.

---

# Missing Flag Representation

Each multimodal sample contains an explicit missingness dictionary:

```python
{
    "photographs_missing": False,
    "radiographs_missing": True,
    "clinical_text_missing": False
}

The flags represent whether each modality is unavailable for the current visit.

Supported Missing Modalities
1. Photographs

Definition:

photographs_missing = True

when no photograph reference exists for the visit.

2. Radiographs

Definition:

radiographs_missing = True

when no radiograph reference exists for the visit.

Radiograph missingness is especially important because it represents the main naturally incomplete modality in the COde dataset.

3. Clinical Text

Definition:

clinical_text_missing = True

when available clinical text fields do not contain usable information.

Implementation

The missing modality handling is implemented using two independent components:

src/data/missing_modality.py

src/data/missing_modality_validation.py

The first component generates missing flags for individual samples.

The second component evaluates missingness statistics across the complete dataset.

Validation Strategy

The validation process:

Loads all dataset visits.
Creates a multimodal sample representation.
Generates missing modality flags.
Aggregates missingness statistics.

The validation is performed at visit level.

Dataset-Level Validation Results

Dataset:

COde Dataset

Total samples:

8775 visits

Validation output:

{
    "total_samples": 8775,
    "photographs_missing": 3,
    "radiographs_missing": 4519,
    "clinical_text_missing": 207
}
Missing Modality Statistics
Modality	Missing Samples	Missing Rate
Photographs	3	0.03%
Radiographs	4519	51.50%
Clinical Text	207	2.36%
Interpretation

The validation confirms that modality availability is highly heterogeneous.

The most significant observation is the radiograph missingness:

51.50% of visits do not contain radiographs.

This confirms that the COde dataset naturally provides a suitable environment for studying:

missing-modality learning,
robust multimodal representation learning,
incomplete multimodal fusion strategies.
Relation to Research Direction

The explicit representation of missing modalities enables future experiments including:

image-only learning,
image-text learning,
complete multimodal learning,
missing-radiograph robustness evaluation,
modality dropout experiments.

The missingness information will be preserved during Dataset and DataLoader implementation.

Reproducibility Considerations

Missing modality detection is implemented as a standalone pipeline component.

Generated outputs:

results/missing_modality/

├── missing_modality_summary.json

The component is independent from:

model architecture,
training strategy,
modality fusion methods.

Therefore, different experiments can reuse the same validated sample representation.

Limitations

At this stage:

missing modalities are only detected,
no missing modality prediction is performed,
no imputation strategy is applied,
no modality reliability weighting is introduced.

These topics belong to later modeling and experimental milestones.

Next Step

The next stage will complete the multimodal dataset foundation by integrating:

sample representation,
image loading,
clinical text loading,
missing modality indicators.

This will prepare the pipeline for the final Dataset and DataLoader implementation.

# Thesis Note 03.7 — PyTorch Dataset Construction

## Overview

This stage implements the PyTorch Dataset layer for the COde multimodal dental dataset.

The objective of this component is to provide a unified dataset interface where each dataset item represents one complete multimodal dental visit.

The Dataset layer integrates previously developed pipeline components:

* Multimodal sample representation
* Image reference handling
* Clinical text loading
* Missing-modality detection

---

# Dataset Design

The implemented dataset follows the principle:


Raw Dataset CSV
|
v
COdeDataset
|
v
MultimodalSample
|
+----------------+
| |
v v
Image References Clinical Text
|
v
Missing Flags


Each returned item corresponds to one dental visit/checkup.

---

# Sample Unit Definition

The dataset unit is defined as:


Sample = Visit / Checkup


Each sample contains:

```python
MultimodalSample(
    patient_id,
    visit_id,
    photographs,
    radiographs,
    clinical_text,
    metadata,
    labels,
    missing_flags
)

Patient-level splitting remains independent from Dataset construction.

All visits belonging to the same patient are assigned to the same split during the patient-level split stage.

Implementation

The Dataset implementation is provided as:

src/data/code_dataset.py

The Dataset class:

COdeDataset

inherits from:

torch.utils.data.Dataset

and implements:

__len__()
__getitem__()
Dataset Responsibilities

The Dataset layer is responsible for:

loading the dataset metadata,
parsing image references,
loading clinical text fields,
constructing multimodal samples,
attaching missing modality indicators.

The Dataset layer does not perform:

image resizing,
image normalization,
augmentation,
label encoding,
model-specific preprocessing.

These operations are reserved for later experiment-specific components.

Returned Sample Example

A single dataset item contains:

{
    "patient_id": "1",

    "visit_id": "0001-001",

    "photographs": [
        "0001-001-01.jpg"
    ],

    "radiographs": [
        "0001-001-01.jpg"
    ],

    "clinical_text": {...},

    "missing_flags": {
        "photographs_missing": False,
        "radiographs_missing": False,
        "clinical_text_missing": False
    }
}
Dataset Validation

A dedicated validation module was implemented:

src/data/dataset_validation.py

The validation verifies:

dataset size,
sample construction,
modality availability,
missing flag consistency.
Validation Results

Dataset:

COde Dataset

Total samples:

8775 visits

Validation output:

{
    "total_samples": 8775,
    "samples_with_photographs": 8772,
    "samples_with_radiographs": 4256,
    "samples_with_clinical_text": 8775
}

Detected errors:

0
Interpretation

The successful validation confirms that the multimodal Dataset layer correctly integrates all previously implemented components.

The dataset can now provide consistent multimodal samples while preserving natural modality incompleteness.

This creates the foundation required for:

multimodal representation learning,
missing-modality experiments,
PyTorch DataLoader integration,
future model training pipelines.
Reproducibility

The Dataset layer is implemented as an independent module:

src/data/code_dataset.py

The validation outputs are stored in:

results/dataset_validation/

└── dataset_validation_summary.json

This modular design allows future experiments to reuse the same validated dataset representation.

Limitations

At this stage:

images are returned as references rather than transformed tensors,
batching is not implemented,
no augmentation is applied,
no task-specific labels are generated.

These components will be introduced in later pipeline and modeling stages.

Next Step

The next stage will implement the DataLoader layer:

batch construction,
variable-length modality handling,
multimodal collation strategy.

This will complete the transition from dataset representation to model-ready input pipeline.

# Thesis Note 03.8 — DataLoader Factory and Multimodal Batching

## Overview

This stage implements the DataLoader layer of the multimodal data pipeline for the COde dental dataset.

The objective of this component is to transform the validated dataset representation into model-ready batches while preserving:

* patient-level split integrity,
* multimodal structure,
* naturally missing modalities,
* variable number of images per visit.

---

# Motivation

The COde dataset contains variable-length multimodal samples.

A single dental visit may include:

* multiple photographs,
* multiple radiographs,
* clinical text,
* missing modalities.

Therefore, the default PyTorch batching mechanism is insufficient because samples cannot be directly stacked into fixed-size tensors at this stage.

A custom batching strategy is required.

---

# DataLoader Architecture

The implemented pipeline follows:


Patient-Level Split
|
v
COdeDataset
|
v
DataLoader Factory
|
v
Custom Collation
|
v
Multimodal Batch


---

# Patient-Level Split Integration

The DataLoader does not perform any dataset splitting.

Instead, it consumes the existing split information generated during the Patient-Level Split stage.

Current dataset distribution:


train:
8175 samples

test:
600 samples


The original patient-level separation is preserved.

---

# DataLoader Factory

The DataLoader creation logic is implemented as:


src/data/dataloader_factory.py


The factory is responsible for:

* creating train DataLoader,
* creating test DataLoader,
* applying batch configuration,
* connecting Dataset and Collate function.

The factory does not perform:

* data preprocessing,
* augmentation,
* label transformation,
* modality fusion.

---

# Configuration Parameters

The DataLoader layer supports configurable parameters:


batch_size
num_workers
shuffle


These parameters are separated from dataset logic and can later be controlled through experiment configuration files.

---

# Custom Multimodal Collation

Because each visit may contain a different number of images, PyTorch default collation cannot be directly applied.

Example:


Visit A:

photographs:
[
image_1,
image_2
]

Visit B:

photographs:
[
image_3
]


The number of images is not fixed.

Therefore, a custom collation function was implemented:


src/data/collate.py


The collator preserves variable-length multimodal information:


Batch

|
+-- patient_id
|
+-- visit_id
|
+-- photographs
|
+-- radiographs
|
+-- clinical_text
|
+-- missing_flags


---

# Validation

A dedicated validation module was implemented:


src/data/dataloader_validation.py


Validation checks:

* DataLoader construction,
* sample count,
* batch generation,
* batch structure consistency.

---

# Validation Results

Validation was performed on the COde dataset.

Configuration:


batch_size = 4


Result:

```json
{
    "train": {
        "split": "train",
        "num_samples": 8175,
        "num_batches": 2044,
        "batch_size": 4,
        "sample_check": true
    },
    "test": {
        "split": "test",
        "num_samples": 600,
        "num_batches": 150,
        "batch_size": 4,
        "sample_check": true
    }
}
Interpretation

The successful validation confirms that the DataLoader layer can correctly generate multimodal batches while maintaining:

patient-level separation,
sample integrity,
missing modality information,
variable image counts.

This completes the transition from raw dataset representation to a training-compatible PyTorch input pipeline.

Reproducibility

Implemented components:

src/data/

├── dataloader_factory.py
├── collate.py
└── dataloader_validation.py

Validation output:

results/dataloader_validation/

└── dataloader_validation_summary.json
Limitations

At this stage:

images are not transformed into model tensors inside Dataset,
augmentation is not applied,
batching does not perform modality fusion,
labels are not task-specific.

These components will be introduced in later modeling stages.

Next Step

The Data Pipeline foundation is now complete.

The next milestone can focus on:

baseline models,
multimodal representation learning,
self-supervised pretraining,
missing-modality experiments.

# Thesis Note 03.9 — Pipeline Validation

## Overview

This stage performs end-to-end validation of the implemented multimodal data pipeline for the COde dental dataset.

The objective is to ensure that all previously developed components work consistently together:

- Patient-level dataset splitting
- Multimodal sample construction
- Image loading
- Clinical text loading
- Missing modality handling
- PyTorch Dataset
- DataLoader generation

The validation is performed before moving to model development stages.

---

# Validation Objectives

The pipeline validation evaluates the following aspects:

1. Dataset sample consistency
2. Modality availability
3. Missing modality preservation
4. Patient-level split correctness
5. Deterministic behavior

---

# Sample Count Validation

The dataset uses:


Sample unit:
Visit / Checkup


The validated dataset contains:


Total samples: 8775 visits


Each sample corresponds to one dental checkup and contains:

- patient identifier,
- visit identifier,
- available modalities,
- clinical information,
- missing modality indicators.

---

# Modality Availability Validation

The availability of each modality was verified.

Validation result:

| Modality | Available Samples |
|---|---:|
| Photographs | 8772 |
| Radiographs | 4256 |
| Clinical Text | 8775 |

The results confirm that:

- photographs are available for almost all visits,
- radiographs have naturally incomplete availability,
- clinical text exists for every visit.

---

# Missing Modality Validation

Missingness is evaluated at the modality level.

A modality is considered missing only when the complete modality is unavailable.

Individual missing fields inside clinical text are not considered modality missingness because other clinical text components may still exist.

Validation result:

| Modality | Missing Samples |
|---|---:|
| Photographs | 3 |
| Radiographs | 4519 |
| Clinical Text | 0 |

The missing radiograph pattern represents a natural characteristic of the dataset.

No imputation or sample removal is applied.

Missing modality information is preserved for future missing-modality experiments.

---

# Patient-Level Split Validation

The dataset split strategy follows:


Split unit:
Patient


All visits belonging to the same patient must remain inside the same split.

The validation checks for patient leakage between splits.

Result:


Patients with split leakage: 0


This confirms that the evaluation protocol is free from patient-level information leakage.

---

# Deterministic Validation

The pipeline configuration enables deterministic execution:

```yaml
project:
  seed: 42
  deterministic: true

A deterministic validation check was performed by repeatedly loading samples and verifying consistent outputs.

Result:

Deterministic:
True

Checked samples:
10
Validation Output

The final validation report is saved as:

results/pipeline_validation/
└── pipeline_validation_report.json

The report contains:

sample statistics,
modality statistics,
missingness statistics,
patient split verification,
deterministic checks.
Conclusion

The multimodal data pipeline successfully passed all validation checks.

The pipeline is now ready for downstream experiments including:

self-supervised representation learning,
multimodal fusion,
missing-modality robustness experiments.

The current implementation preserves the original characteristics of the COde dataset without removing or artificially completing missing information.

# 03.10 — Data Pipeline Runner

## Objective

This notebook provides a lightweight execution interface for the implemented multimodal data pipeline.

The purpose of this runner is to verify that all pipeline components can be executed together:

- Dataset loading
- Patient-level split usage
- DataLoader construction
- Multimodal sample inspection
- Missing modality analysis

The notebook contains only execution and visualization logic.
All data processing components are implemented inside the `src/data` modules.

In [5]:
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent


if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )


print("Project root:")
print(PROJECT_ROOT)

Project root:
/home/ubuntu/Projects/thesis-code


In [3]:
from src.data.code_dataset import COdeDataset
from src.data.dataloader_factory import DataLoaderFactory

## Dataset Initialization

In [8]:
dataset_path = (
    PROJECT_ROOT
    / "data/raw/COde-Dataset/complete_dataset.csv"
)


dataset = COdeDataset(
    str(dataset_path)
)

print(len(dataset))

8775


## DataLoader Construction

In [9]:
factory = DataLoaderFactory(
    dataset,
    batch_size=4,
)


train_loader = factory.create_train_loader()

test_loader = factory.create_test_loader()


print(
    "Train samples:",
    len(train_loader.dataset)
)

print(
    "Test samples:",
    len(test_loader.dataset)
)

Train samples: 8175
Test samples: 600


## Multimodal Sample Inspection

In [10]:
sample = dataset[0]


print("Patient ID:")
print(sample.patient_id)

print()

print("Visit ID:")
print(sample.visit_id)

print()

print("Photographs:")
print(len(sample.photographs))

print()

print("Radiographs:")
print(len(sample.radiographs))

print()

print("Clinical text fields:")
print(len(sample.clinical_text))

print()

print("Missing flags:")
print(sample.missing_flags)

Patient ID:
1

Visit ID:
0001-001

Photographs:
1

Radiographs:
1

Clinical text fields:
11

Missing flags:
{'photographs_missing': False, 'radiographs_missing': False, 'clinical_text_missing': False}


## Dataset Split Statistics

In [11]:
import pandas as pd


df = pd.read_csv(
    dataset_path
)


df["split"].value_counts()

split
train    8175
test      600
Name: count, dtype: int64

# Conclusion

The Data Pipeline Runner successfully verifies the complete multimodal data loading workflow.

The pipeline can now:

- construct patient-level samples,
- load image modalities,
- load clinical text,
- preserve missing modality information,
- generate PyTorch DataLoaders.

This completes the Data Pipeline milestone.

The next milestone will focus on model development and baseline experiments.